# SantéNLP: fine-tune Bio_ClinicalBERT on Colab

Trains the specialty classifier (`src/classify.py`) on a GPU and saves the results to Google Drive.

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Then run the cells from top to bottom.

What gets trained, for each of the 2 ways of shortening long reports (`head`, `head_tail`):
a main model on the saved split + 5 seeds (new train/val split each, same test set) = 12 fine-tunings in total.

Results go to `MyDrive/SanteNLP/`:
- `outputs/runs/<date>_bert_specialty_<strategy>/`: config, metrics, test predictions (row numbers only)
- `models/bert_specialty/<strategy>/`: the main model of each strategy (~420 MB each)

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys
print("Python", sys.version)

## 2. Connect Google Drive

A window asks for permission: accept it. Results are written to Drive so they survive the end of the Colab session.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/SanteNLP"
!mkdir -p "{DRIVE}/outputs/runs" "{DRIVE}/models"
print("Results folder:", DRIVE)

## 3. Get the code

In [ ]:
REPO = "https://github.com/mohcine00errachind/SanteNLP.git"
!rm -rf /content/SanteNLP && git clone --quiet --depth 1 {REPO} /content/SanteNLP
%cd /content/SanteNLP
!git log -1 --format="Code version: %h  %s"  

## 4. Install the pinned libraries

Colab already has a GPU build of PyTorch, so `torch` is **not** reinstalled. Also skipped: the spaCy word vectors
(NER only), notebook tools (Colab has its own) and Streamlit (app only).

pip may print *dependency conflict* warnings about Colab's own pre-installed packages: they are expected and
harmless here, because training runs in a separate Python process (`!python ...`).

In [ ]:
!grep -v -E "^(torch|en_core_web_md|ipykernel|nbformat|nbclient|streamlit)\b" requirements.txt > /tmp/requirements_colab.txt
!pip install -q -r /tmp/requirements_colab.txt
!python -c "import torch, transformers; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| transformers', transformers.__version__)" 

## 5. Download and prepare the data

Same script as locally. The check below makes sure Colab built **exactly** the same train/val/test split.

In [ ]:
!python -m src.data --config configs/data.yaml

In [ ]:
from src.data import fingerprint, load_mtsamples_splits

# Fingerprints of the split built locally (Windows), see src/data.py
EXPECTED = {"train": 5892761080162288042, "val": 6317211258887222346, "test": 1618043657115009311}
splits = load_mtsamples_splits()
for name, df in splits.items():
    assert fingerprint(df) == EXPECTED[name], f"{name} split differs from the local one!"
print("Same split as locally:", {name: len(df) for name, df in splits.items()})

## 6. Quick check (about 1 minute)

A tiny run (24 reports, 1 epoch) to catch errors before the long training. Its scores mean nothing.

In [ ]:
!python -m src.classify --smoke --runs-dir /tmp/smoke

## 7. Train: `head` strategy (first 510 tokens)

6 fine-tunings (main model + 5 seeds). Each line printed is one finished model.
If the session disconnects, just rerun from step 2: finished runs are already on Drive.

In [ ]:
!python -m src.classify --strategies head --runs-dir "{DRIVE}/outputs/runs" --model-dir "{DRIVE}/models/bert_specialty" 

## 8. Train: `head_tail` strategy (first 128 + last 382 tokens)

In [ ]:
!python -m src.classify --strategies head_tail --runs-dir "{DRIVE}/outputs/runs" --model-dir "{DRIVE}/models/bert_specialty" 

## 9. Summary

In [ ]:
import glob, json

print(f"{'run':45s} {'main test F1':>12s} {'seeds: mean ± std':>20s}")
for path in sorted(glob.glob(f"{DRIVE}/outputs/runs/*_bert_specialty_*")):
    m = json.load(open(f"{path}/metrics.json"))
    print(f"{path.split('/')[-1]:45s} {m['main']['test']['macro_f1']:12.3f} "
          f"{m['test_macro_f1_mean']:13.3f} ± {m['test_macro_f1_std']:.3f}")
print("\nBaseline (TF-IDF + logistic regression): main 0.954, seeds 0.940 ± 0.003")

## 10. Next step

Copy these two Drive folders into the project on your computer (see the README):
- `MyDrive/SanteNLP/outputs/runs/*_bert_specialty_*` → `outputs/runs/`
- `MyDrive/SanteNLP/models/bert_specialty/` → `models/bert_specialty/`